# GigaChat v2 by example: what changes from v1

Read this notebook as a short cookbook: **task → short code example → model response**.
After setup, you can run each example separately. Responses appear directly below the cells.
Here, v1 and v2 refer to API routes in the same version of `langchain-gigachat`, not LangChain versions.

Start with a simple question and compare the responses, then try streaming, JSON, tools, reasoning, and files.
Calls consume tokens. Later examples upload an existing JPG and create a conversation that continues by ID.

**How to read the output:** the output below a code cell is the actual response from that run. An API error means
that the example did not succeed on the selected endpoint/model; it is never replaced with an invented response.

> Blocks labeled "Illustrative response" show the expected output format. Actual responses appear below executed cells. Run the generation, upload, and conversation-continuation examples separately.

## Setup

Use SDK `0.2.4a1` with the `langchain-gigachat 0.5.2a1` prerelease.
The commands below assume the SDK source is in `../gigachat`; adjust the path if needed.
From the repository root:

```bash
python -m venv .venv-notebook
source .venv-notebook/bin/activate
python -m pip install -e ../gigachat -e ./libs/gigachat jupyterlab ipykernel python-dotenv
python -m ipykernel install --user --name gigachat-notebook --display-name "GigaChat notebook"
python -m jupyterlab examples/api_v2_feature_comparison.ipynb
```

Select the **GigaChat notebook** kernel. Set `GIGACHAT_CREDENTIALS` or `GIGACHAT_ACCESS_TOKEN` in `.env`
(the `GIGACHAT_USER` / `GIGACHAT_PASSWORD` pair is also supported). Secrets are not printed.
Set `GIGACHAT_BASE_URL` and `GIGACHAT_CA_BUNDLE_FILE` if needed.
The `/v1` base URL is intentional: the SDK selects the v2 route itself.

After setup, run the cells you want to try; there is no separate RUN_LIVE switch.

In [ ]:
import os
import json
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_gigachat import GigaChat

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "libs/gigachat/pyproject.toml").exists())
load_dotenv(ROOT / ".env", override=False)
MODEL = os.getenv("GIGACHAT_V2_EXAMPLE_MODEL", os.getenv("GIGACHAT_MODEL", "GigaChat-3-Ultra"))
BASE_URL = os.getenv("GIGACHAT_BASE_URL", "https://api.giga.chat/v1")
AUTH = {key: os.getenv("GIGACHAT_" + key.upper()) for key in (
    "credentials", "access_token", "user", "password", "scope", "auth_url",
    "ca_bundle_file", "cert_file", "key_file", "key_file_password")}
AUTH = {key: value for key, value in AUTH.items() if value}

def model(v2=True, **options):
    settings = dict(model=MODEL, base_url=BASE_URL, use_api_v2=v2,
                    temperature=0, max_tokens=512, timeout=45, max_retries=0, **AUTH)
    settings.update(options)
    return GigaChat(**settings)

v1 = model(v2=False)
v2 = model(v2=True)
print(f"Model: {MODEL}")
print(f"langchain-gigachat {version('langchain-gigachat')}; SDK {version('gigachat')}")

### Two small helpers for readable output

`answer()` displays the message text; `details()` shows useful metadata without dumping the entire object.
All model calls below are ordinary Python code, without a test runner.

In [ ]:
def answer(message, title="Response"):
    display(Markdown(f"**{title}**"))
    display(Markdown(message.text or "_No text response; inspect tool_calls/content_blocks._"))

def details(message):
    print("Finish reason:", message.response_metadata.get("finish_reason"))
    print("Tokens:", message.usage_metadata)

def collect(chunks):
    full = None
    for chunk in chunks:
        print(chunk.text, end="", flush=True)
        full = chunk if full is None else full + chunk
    print()
    if full is None:
        raise RuntimeError("The server returned an empty stream")
    return full

## 1. One question, two responses

Ask for an explanation of idempotence using an everyday example. The prompt, model, and sampling settings are identical.
Run v1 first, then v2, and read both responses. The text does not have to match.

In [ ]:
question = "Explain idempotence using an elevator button as an example. Use no more than three sentences."
response_v1 = v1.invoke(question)
answer(response_v1, "API v1")
details(response_v1)

In [ ]:
response_v2 = v2.invoke("Explain idempotence using an elevator button as an example. Use no more than three sentences.")
answer(response_v2, "API v2")
details(response_v2)

**Illustrative response**

Pressing the elevator call button several times has the same effect as pressing it once: the elevator has already been called. Pressing it again does not create additional trips.

**What changed in the code:** only `use_api_v2=True`. The result is still an `AIMessage`.
You can also switch the route for a single call using `bind`.

In [ ]:
one_off = v1.bind(use_api_v2=True).invoke("Complete the sentence: repeating an idempotent operation…")
answer(one_off)

## 2. Display the response as it is generated

`stream()` yields chunks. Print the text immediately and combine the chunks to preserve the final message.
The two examples below are independent and use the same prompt.

In [ ]:
stream_v1 = collect(v1.stream("Give three short tips for writing clear Python functions."))
details(stream_v1)

In [ ]:
stream_v2 = collect(v2.stream("Give three short tips for writing clear Python functions."))
details(stream_v2)
print("Model in the final metadata:", stream_v2.response_metadata.get("model"))

**Illustrative text shown progressively**

1. Give functions names that describe what they do.
2. Give each function one clear responsibility.
3. Show what a function does with a short example.

## 3. An asynchronous request

Jupyter supports `await` directly in a cell. You do not need `asyncio.run()` here.

In [ ]:
async_v1 = await v1.ainvoke("Suggest a short name for an app that tracks houseplants.")
answer(async_v1, "API v1 · ainvoke")

In [ ]:
async_v2 = await v2.ainvoke("Suggest a short name for an app that tracks houseplants.")
answer(async_v2, "API v2 · ainvoke")

**Illustrative response**

"Little Leaf"

### Asynchronous streaming

Use the same approach, but receive chunks with `async for`.

In [ ]:
async_stream_v1 = None
async for chunk in v1.astream("Write a two-line message congratulating a team on completing a project."):
    print(chunk.text, end="", flush=True)
    async_stream_v1 = chunk if async_stream_v1 is None else async_stream_v1 + chunk
print()
details(async_stream_v1)

In [ ]:
async_stream_v2 = None
async for chunk in v2.astream("Write a two-line message congratulating a team on completing a project."):
    print(chunk.text, end="", flush=True)
    async_stream_v2 = chunk if async_stream_v2 is None else async_stream_v2 + chunk
print()
details(async_stream_v2)

## 4. Get an object instead of free-form text

Extract a product and quantity from a short sentence. Pydantic defines the result structure.
`json_schema` is a native API format; it was already available in v1 and uses a new mapping in v2.
Support depends on the model.

In [ ]:
class Product(BaseModel):
    """A product from a customer message."""
    name: str = Field(description="Product name in the singular")
    count: int = Field(description="Number of units")

order = "Add 3 chairs to the order."

In [ ]:
product_v1 = v1.with_structured_output(Product, method="json_schema").invoke(order)
print(product_v1.model_dump_json(indent=2))
print("Quantity as a Python int:", product_v1.count)

In [ ]:
product_v2 = v2.with_structured_output(Product, method="json_schema", strict=True).invoke(order)
print(product_v2.model_dump_json(indent=2))
print("Quantity as a Python int:", product_v2.count)

**Illustrative object**

```json
{
  "name": "chair",
  "count": 3
}
```

### Get both the raw response and the parsed object

`include_raw=True` returns `raw`, `parsed`, and `parsing_error`. This is useful when debugging a schema.

In [ ]:
structured = v2.with_structured_output(Product, method="json_schema", include_raw=True).invoke(order)
answer(structured["raw"], "Raw JSON from the model")
print("Parsed object:", structured["parsed"])
print("Parsing error:", structured["parsing_error"])

### The same object through function calling

This method uses a client function to provide the structure. Compare it with native JSON Schema.

In [ ]:
function_product_v1 = v1.with_structured_output(Product, method="function_calling").invoke(order)
print(function_product_v1.model_dump_json(indent=2))

In [ ]:
function_product_v2 = v2.with_structured_output(Product, method="function_calling").invoke(order)
print(function_product_v2.model_dump_json(indent=2))

### A JSON object without a user-provided schema (v2)

If the fields are not known in advance, pass `None` and `method="json_mode"`. The wrapper sends `response_format={"type": "json_schema", "schema": {"type": "object"}}`: the API contract requires `schema`, and the minimal schema allows arbitrary object fields. `strict` is omitted. The result is parsed into a Python `dict`; arrays and scalars cause a parsing error.

In [ ]:
json_object = v2.with_structured_output(None, method="json_mode").invoke(
    "Return a JSON object with a short answer to the question: what is JSON?"
)
print(json.dumps(json_object, ensure_ascii=False, indent=2))
print("Result type:", type(json_object).__name__)

### Low-level response_format

`bind(response_format=...)` controls the request format but returns a regular `AIMessage`.
For a parsed Pydantic object, use `with_structured_output()` as shown above.

In [ ]:
raw_json = v2.bind(response_format={
    "type": "json_schema", "schema": Product.model_json_schema(), "strict": True,
}).invoke(order)
answer(raw_json)
print("Result type:", type(raw_json).__name__)
print("Manual parsing:", Product.model_validate_json(raw_json.text))

## 5. A tool that actually runs in Python

The model will request multiplication of 17 by 23. Python will perform the calculation and return **391**.
The model then receives a `ToolMessage` and writes a response. You can inspect each of these three separate steps.

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

### Step 1. The model selects the function and arguments (v2)

In [ ]:
tool_question = HumanMessage("Calculate 17 × 23 using multiply and explain the result in one sentence.")
tool_request = v2.bind_tools([multiply], tool_choice="multiply").invoke([tool_question])
print(json.dumps(tool_request.tool_calls, ensure_ascii=False, indent=2))
print("Server returned tools_state_id:", bool(tool_request.additional_kwargs.get("tools_state_id")))

**Illustrative tool request**

```python
[{"name": "multiply", "args": {"a": 17, "b": 23}, "id": "<server-provided ID>", "type": "tool_call"}]
```
The ID above is a placeholder; copy the ID from the actual response into `ToolMessage`.

### Step 2. Execute the requested function

Copy the call ID unchanged. Keep the original `tool_request`: it contains the provider's state.

Pass the result as a JSON object: `json.dumps({"result": calculated})`.
This example does not use `str(calculated)`: the adapter parses `"391"` as a number,
and the endpoint in a previous user run returned 422 for that tool result.
The object matches the result format in the integration examples; this correction still needs a live rerun.

In [ ]:
call = tool_request.tool_calls[0]
if call["name"] != "multiply":
    raise ValueError("Unexpected tool")
calculated = multiply.invoke(call["args"])
print("Python result:", calculated)
tool_result = ToolMessage(content=json.dumps({"result": calculated}), tool_call_id=call["id"], name=call["name"])

**Illustrative local calculation for a=17, b=23**

`391` is the result of the Python function, not generated model text.

### Step 3. Return the result to the model

In [ ]:
tool_final = v2.bind_tools([multiply], tool_choice="none").invoke(
    [tool_question, tool_request, tool_result])
answer(tool_final)

**Illustrative final response**

17 × 23 = 391.

### For comparison: the full cycle on v1

The public code is almost identical. Start a new history: v2 `tools_state_id` cannot be passed to v1,
and legacy `functions_state_id` cannot be passed to v2.

In [ ]:
legacy_question = HumanMessage("Calculate 17 × 23 using multiply and explain the result in one sentence.")
legacy_request = v1.bind_tools([multiply], tool_choice="multiply").invoke([legacy_question])
print("Tool request:", legacy_request.tool_calls)
legacy_call = legacy_request.tool_calls[0]
if legacy_call["name"] != "multiply":
    raise ValueError("Unexpected tool")
legacy_value = multiply.invoke(legacy_call["args"])
print("Python result:", legacy_value)
legacy_result = ToolMessage(content=json.dumps({"result": legacy_value}), tool_call_id=legacy_call["id"], name=legacy_call["name"])
legacy_final = v1.bind_tools([multiply], tool_choice="none").invoke(
    [legacy_question, legacy_request, legacy_result])
answer(legacy_final, "API v1 · final response")

### A streamed tool call (v2)

Combine the chunks first: an individual chunk may contain only part of the arguments.
Once assembled, continue the cycle with a regular `ToolMessage`.

In [ ]:
stream_question = HumanMessage("Calculate 17 × 23 using multiply.")
stream_request = collect(v2.bind_tools([multiply], tool_choice="multiply").stream([stream_question]))
print("Assembled tool call:", stream_request.tool_calls)
stream_call = stream_request.tool_calls[0]
if stream_call["name"] != "multiply":
    raise ValueError("Unexpected tool")
stream_value = multiply.invoke(stream_call["args"])
print("Python result:", stream_value)
stream_result = ToolMessage(content=json.dumps({"result": stream_value}), tool_call_id=stream_call["id"], name=stream_call["name"])
answer(v2.bind_tools([multiply], tool_choice="none").invoke(
    [stream_question, stream_request, stream_result]))

## 6. A reasoning model

Reasoning was already supported in v1. Here we compare responses to the same problem and check
whether reasoning blocks appear in the new contract. Set the model with `GIGACHAT_V2_REASONING_MODEL`.

In [ ]:
REASONING_MODEL = os.getenv("GIGACHAT_V2_REASONING_MODEL", "GigaChat-2-Reasoning")
reasoning_question = "Anna has 12 coins, and Boris has twice as many. Together they give away 9 coins. How many are left?"
think_v1 = model(v2=False, model=REASONING_MODEL, reasoning_effort="high", max_tokens=2048)
think_v2 = model(v2=True, model=REASONING_MODEL, reasoning_effort="high", max_tokens=2048)

In [ ]:
reasoning_v1 = think_v1.invoke(reasoning_question)
answer(reasoning_v1, "v1 · answer to the problem")
print("Reasoning in additional_kwargs:", bool(reasoning_v1.additional_kwargs.get("reasoning_content")))
print("Block types:", [b.get("type") for b in reasoning_v1.content_blocks])

In [ ]:
reasoning_v2 = think_v2.invoke(reasoning_question)
answer(reasoning_v2, "v2 · answer to the problem")
print("Reasoning in additional_kwargs:", bool(reasoning_v2.additional_kwargs.get("reasoning_content")))
print("Block types:", [b.get("type") for b in reasoning_v2.content_blocks])

**Illustrative answer to the problem**

Boris has 24 coins. Together they had 12 + 24 = 36 coins, and after giving away 9, they have 27 left.

### Reasoning with streaming

In [ ]:
reasoning_stream_v1 = collect(think_v1.stream(reasoning_question))
print("Reasoning received:", bool(reasoning_stream_v1.additional_kwargs.get("reasoning_content"))
      or any(b.get("type") == "reasoning" for b in reasoning_stream_v1.content_blocks))
details(reasoning_stream_v1)

In [ ]:
reasoning_stream_v2 = collect(think_v2.stream(reasoning_question))
print("Reasoning received:", bool(reasoning_stream_v2.additional_kwargs.get("reasoning_content"))
      or any(b.get("type") == "reasoning" for b in reasoning_stream_v2.content_blocks))
details(reasoning_stream_v2)

## 7. Generation settings in the new contract

The adapter maps the usual `temperature`, `max_tokens`, and `reasoning_effort` arguments to `model_options`.
You can also pass native options, which take precedence over the regular arguments.
Here we also enable the filter and disable the ranker. This demonstrates how to pass settings, rather than measuring their effect.

In [ ]:
configured = v2.invoke(
    "Suggest three short names for a book club.",
    model_options={"temperature": 0.7, "top_p": 0.9, "max_tokens": 128,
                   "repetition_penalty": 1.0, "update_interval": 0.5},
    ranker_options={"enabled": False}, profanity_check=True,
)
answer(configured)

**Illustrative response**

1. Between the Lines
2. Reading Together
3. The Bookshelf

## 8. Server-side web search

`web_search` runs on the GigaChat server. This binding is not available in v1.
In addition to the text, inspect sources and tool events if the server returns them.

In [ ]:
search = v2.bind_tools([{"type": "web_search"}], tool_choice="web_search")
search_answer = search.invoke("Find the official Python website. Briefly explain what kind of language Python is and cite the source.")
answer(search_answer)
for block in search_answer.content_blocks:
    for source in block.get("annotations", []):
        print("Source:", source)
    if block.get("type") in ("server_tool_call", "server_tool_result"):
        print("Tool:", block.get("name"), "event:", block["type"], "status:", block.get("status"))

**Illustrative response with a source**

Python is a general-purpose programming language. Its official website is https://www.python.org/.

The actual output displays server-returned annotations separately. This illustrative response does not confirm that web_search ran.

### Search with streaming

Server tool blocks are preserved in the aggregated message alongside the text.

In [ ]:
search_stream = collect(v2.bind_tools([{"type": "web_search"}], tool_choice="web_search").stream(
    "Find the official Python website and provide a link."))
print("Result blocks:", [b.get("type") for b in search_stream.content_blocks])
details(search_stream)

## 9. Server-side code interpreter

Ask for the sum of squares from 1 to 20. We can verify locally that the result is 2870.
This differs from our Python `multiply` tool: the provider is asked to perform the calculation.

In [ ]:
interpreter = v2.bind_tools([{"type": "code_interpreter"}], tool_choice="code_interpreter")
code_answer = interpreter.invoke("Use code_interpreter to calculate the sum of squares of the integers from 1 to 20.")
answer(code_answer)
print("Local Python result for comparison:", sum(i*i for i in range(1, 21)))
print("Result blocks:", [b.get("type") for b in code_answer.content_blocks])

**Illustrative expected mathematical result**

The sum of squares from 1 to 20 is **2870**. The presence of this number in the text alone does not prove that code_interpreter ran; inspect the server tool blocks.

### Code interpreter with streaming

In [ ]:
code_stream = collect(v2.bind_tools([{"type": "code_interpreter"}], tool_choice="code_interpreter").stream(
    "Use code_interpreter to calculate the sum of squares of the integers from 1 to 20."))
print("Result blocks:", [b.get("type") for b in code_stream.content_blocks])

## 10. Generate an image with image_generate

GigaChat's built-in tool creates an image on the server. This is a separate, optional example:
a small JPG is already included for the upload in the next section.
The availability of `image_generate` depends on the endpoint and model. First obtain a response containing a file, then download it for display.

In [ ]:
image_reply = v2.bind_tools(
    [{"type": "image_generate"}], tool_choice="image_generate",
).invoke("Draw a yellow mug and two red apples on a light blue table. Do not include any text.")
answer(image_reply, "image_generate response")
print("Response blocks:", [block.get("type") for block in image_reply.content_blocks])
generated_image_ids = [
    block["file_id"] for block in image_reply.content_blocks
    if block.get("type") in ("image", "file") and block.get("file_id")
]
print("Generated file IDs:", generated_image_ids)

### Display the generated image

`get_file_content()` returns base64. Decode it and display the image directly in the notebook.
If the server does not return a file ID, inspect `image_reply.content_blocks`; a text response alone does not confirm image generation.

In [ ]:
import base64
from IPython.display import Image as NotebookImage

if not generated_image_ids:
    raise RuntimeError("image_generate did not return an image file; inspect content_blocks above")
generated_file = v2.get_file_content(generated_image_ids[0])
display(NotebookImage(data=base64.b64decode(generated_file.content), width=384))

## 11. Upload a small JPG and ask what it shows

The file `examples/assets/mug-apples.jpg` is included alongside the notebook: **384 × 384 pixels**,
a yellow mug and two red apples. This is a synthetic image for the example.
Preview it locally, then upload it through the Files API and pass the returned ID to the model.
You do not need to generate the image again for this example.

In [ ]:
from IPython.display import Image as NotebookImage

jpg_path = ROOT / "examples" / "assets" / "mug-apples.jpg"
print(f"File: {jpg_path.name}; size: {jpg_path.stat().st_size / 1024:.1f} KB")
display(NotebookImage(filename=str(jpg_path), width=384))

### Upload the file once

This cell sends the included JPG to the configured GigaChat endpoint. `uploaded.id_` is the file ID,
which you can reuse in requests. Running the upload again creates a new upload.

In [ ]:
with jpg_path.open("rb") as image_file:
    uploaded = v2.upload_file((jpg_path.name, image_file.read()), purpose="general")
image_file_id = uploaded.id_
print("Uploaded file:", image_file_id)

### Describe the image (v2)

In [ ]:
image_question = HumanMessage(content=[
    {"type": "text", "text": "What is in the image? Name the objects, their colors, and the number of apples."},
    {"type": "image", "file_id": image_file_id, "mime_type": "image/jpeg"},
])
image_answer_v2 = v2.invoke([image_question])
answer(image_answer_v2, "Image description · v2")

### The same image (v1)

Reuse the uploaded file; a second upload is unnecessary. For manual comparison: the mug is yellow, and there are two apples.

In [ ]:
image_answer_v1 = v1.invoke([image_question])
answer(image_answer_v1, "Image description · v1")

## 12. Specify the model once, then use only the conversation ID

In this SDK, the identifier is called **`thread_id`**. Read it from the first response;
`message_id` identifies an individual message and is not used to continue the conversation.
You do not need a preconfigured assistant or an ID from `.env` for this example.

The first request passes `model=MODEL` and `storage=True` so that the server stores the conversation.
Subsequent requests pass only the new message and `storage={"thread_id": conversation_id}`.
The adapter omits the model configured in the constructor when continuing by thread ID.

In [ ]:
first_turn = v2.invoke(
    "We are planning a picnic for three people. One person does not eat dairy products. Remember these constraints.",
    model=MODEL,
    storage=True,
)
answer(first_turn, "First response")
conversation_id = first_turn.response_metadata.get("thread_id")
if not conversation_id:
    raise RuntimeError("The server did not return thread_id; cannot continue without an ID")
print("Conversation ID:", conversation_id)

### Second request: omit the model and history

In [ ]:
second_turn = v2.invoke(
    "Suggest a menu for our picnic that respects the dietary restriction. How many people does it serve?",
    storage={"thread_id": conversation_id},
)
answer(second_turn, "Continuation by ID")

### Another request with the same ID, now using streaming

The new message contains neither the number of people nor the dietary restriction. The response should use those details from the stored conversation.

In [ ]:
third_turn = collect(v2.stream(
    "Create a short shopping list for this menu.",
    storage={"thread_id": conversation_id},
))

## 13. A small LangChain chain

Prompt template → model → string parser. This is useful when you want text instead of an `AIMessage`.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an editor. Rewrite the text politely and concisely, without bureaucratic language."),
    ("human", "{text}"),
])
source_text = "You are required to ensure the submission of the report prior to Friday."

In [ ]:
chain_v1 = prompt | v1 | StrOutputParser()
edited_v1 = chain_v1.invoke({"text": source_text})
print("Original:", source_text)
print("v1:", edited_v1)

In [ ]:
chain_v2 = prompt | v2 | StrOutputParser()
edited_v2 = chain_v2.invoke({"text": source_text})
print("Original:", source_text)
print("v2:", edited_v2)

## What to remember about v1 and v2

| Scenario | v1 | v2 |
|---|---|---|
| Standard, async, and streaming requests | Use the legacy SDK | Same LangChain interface, new SDK resources |
| Structured output | function_calling and json_schema | function_calling and native json_schema |
| Client tools | functions / functions_state_id | tools / tools_state_id; preserve the call ID unchanged |
| Reasoning | Already supported | New mapping and stream conversion |
| web_search / code_interpreter / image_generate | No such binding | Built-in server tools, if available on the endpoint |
| Continue by ID | — | Specify the model in the first request; then use storage with the returned thread_id |
| Upload a JPG | upload_file → file ID → message | Same file ID, primary image block |

V2 supports multiple client tool calls with separate IDs, `parallel_tool_calls=True`,
and `tool_choice="any"` / `"required"`. Return a separate `ToolMessage` with the matching ID for each call.
Do not mix tool histories between routes. Parallel server tools without distinguishable IDs are not supported.
See the [tool examples](sdk_contract_alignment/README.md) for complete cycles.

**This is a functional comparison.** We inspect responses and result structures using the same examples.
It is not a speed or quality benchmark; those require repeated runs and a separate methodology.

### Example sources

- [LangChain: models, streaming, tools, and structured output](https://docs.langchain.com/oss/python/langchain/models): recipes checked through Context7.
- [LangChain: structured output](https://docs.langchain.com/oss/python/langchain/structured-output).
- [CHANGELOG 0.5.2a1](../libs/gigachat/CHANGELOG.md) and [API v2 README](../libs/gigachat/README.md#api-v2-v2chatcompletions).
- [Adapter contract tests](../libs/gigachat/tests/unit_tests/chat_models/primary): detailed assertions and protocol errors.

Before publishing, clear responses that contain data from your files or private conversations.